# 06 — DNN Optimization for Customer Churn

Second-stage optimization after `05_DNN_Model_Proper.ipynb`.

The test set is kept untouched until the final evaluation. Model architecture,
hyperparameters, class weighting, and classification threshold are selected
using the training/validation data only.

In [ ]:
import os, random, time, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import tensorflow as tf

from tensorflow.keras import layers, models, callbacks, regularizers
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, roc_curve, classification_report
)

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)

: 

## 1. Load the same data and reproduce the existing preprocessing

In [ ]:
data_path = os.path.join("..", "data", "WA_Fn-UseC_-Telco-Customer-Churn.csv")
if not os.path.exists(data_path):
    raise FileNotFoundError(f"Dataset not found: {data_path}")

df = pd.read_csv(data_path)
df = df.drop("customerID", axis=1)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

X = df.drop("Churn", axis=1)
y = df["Churn"].map({"No": 0, "Yes": 1})

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)
X_train_tr, X_val, y_train_tr, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.20,
    random_state=SEED, stratify=y_train_full
)

numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), numeric_features),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", drop="first"))
    ]), categorical_features)
])

X_train_p = preprocessor.fit_transform(X_train_tr)
X_val_p = preprocessor.transform(X_val)
X_test_p = preprocessor.transform(X_test)

if hasattr(X_train_p, "toarray"):
    X_train_p = X_train_p.toarray()
    X_val_p = X_val_p.toarray()
    X_test_p = X_test_p.toarray()

X_train_np = np.asarray(X_train_p, dtype=np.float32)
X_val_np = np.asarray(X_val_p, dtype=np.float32)
X_test_np = np.asarray(X_test_p, dtype=np.float32)

y_train_np = y_train_tr.to_numpy(dtype=np.float32)
y_val_np = y_val.to_numpy(dtype=np.float32)
y_test_np = y_test.to_numpy(dtype=np.float32)

print("Train:", X_train_np.shape)
print("Validation:", X_val_np.shape)
print("Test:", X_test_np.shape)

## 2. Flexible DNN functions

In [ ]:
def build_dnn(input_dim, units=(64, 32, 16), dropout=0.3,
              lr=1e-3, optimizer_name="adam", batch_norm=False, l2_rate=0.0):
    model = models.Sequential([layers.Input(shape=(input_dim,))])

    for n in units:
        model.add(layers.Dense(
            n, activation=None,
            kernel_regularizer=regularizers.l2(l2_rate) if l2_rate else None
        ))
        if batch_norm:
            model.add(layers.BatchNormalization())
        model.add(layers.Activation("relu"))
        if dropout:
            model.add(layers.Dropout(dropout))

    model.add(layers.Dense(1, activation="sigmoid"))

    if optimizer_name == "adam":
        opt = tf.keras.optimizers.Adam(learning_rate=lr)
    elif optimizer_name == "rmsprop":
        opt = tf.keras.optimizers.RMSprop(learning_rate=lr)
    else:
        raise ValueError("Use 'adam' or 'rmsprop'.")

    model.compile(
        optimizer=opt,
        loss="binary_crossentropy",
        metrics=[
            tf.keras.metrics.BinaryAccuracy(name="accuracy"),
            tf.keras.metrics.Precision(name="precision"),
            tf.keras.metrics.Recall(name="recall"),
            tf.keras.metrics.AUC(name="roc_auc")
        ]
    )
    return model


def train_model(model, class_weight=None, batch_size=32):
    es = callbacks.EarlyStopping(
        monitor="val_loss", patience=10,
        restore_best_weights=True, verbose=0
    )
    start = time.time()
    history = model.fit(
        X_train_np, y_train_np,
        validation_data=(X_val_np, y_val_np),
        epochs=100, batch_size=batch_size,
        class_weight=class_weight,
        callbacks=[es], verbose=0
    )
    return model, history, time.time() - start


def probabilities(model, X_data):
    return model.predict(X_data, verbose=0).ravel()


def metrics(y_true, prob, threshold=0.5):
    pred = (prob >= threshold).astype(int)
    return {
        "accuracy": accuracy_score(y_true, pred),
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred, zero_division=0),
        "f1": f1_score(y_true, pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, prob)
    }

## 3. Optimization experiments

In [ ]:
experiments = [
    {"name":"O1-64-32", "units":(64,32), "dropout":0.3, "lr":1e-3, "batch":32, "opt":"adam", "bn":False, "cw":False, "l2":0},
    {"name":"O2-128-64-32", "units":(128,64,32), "dropout":0.3, "lr":1e-3, "batch":32, "opt":"adam", "bn":False, "cw":False, "l2":0},
    {"name":"O3-128-64-32-16", "units":(128,64,32,16), "dropout":0.3, "lr":1e-3, "batch":32, "opt":"adam", "bn":False, "cw":False, "l2":0},
    {"name":"O4-Dropout-0.1", "units":(64,32,16), "dropout":0.1, "lr":1e-3, "batch":32, "opt":"adam", "bn":False, "cw":False, "l2":0},
    {"name":"O5-Dropout-0.3", "units":(64,32,16), "dropout":0.3, "lr":1e-3, "batch":32, "opt":"adam", "bn":False, "cw":False, "l2":0},
    {"name":"O6-Dropout-0.4", "units":(64,32,16), "dropout":0.4, "lr":1e-3, "batch":32, "opt":"adam", "bn":False, "cw":False, "l2":0},
    {"name":"O7-LR-5e-4", "units":(64,32,16), "dropout":0.3, "lr":5e-4, "batch":32, "opt":"adam", "bn":False, "cw":False, "l2":0},
    {"name":"O8-LR-3e-4", "units":(64,32,16), "dropout":0.3, "lr":3e-4, "batch":32, "opt":"adam", "bn":False, "cw":False, "l2":0},
    {"name":"O9-LR-1e-4", "units":(64,32,16), "dropout":0.3, "lr":1e-4, "batch":32, "opt":"adam", "bn":False, "cw":False, "l2":0},
    {"name":"O10-BatchNorm", "units":(64,32,16), "dropout":0.2, "lr":1e-3, "batch":32, "opt":"adam", "bn":True, "cw":False, "l2":0},
    {"name":"O11-RMSprop", "units":(64,32,16), "dropout":0.3, "lr":1e-3, "batch":32, "opt":"rmsprop", "bn":False, "cw":False, "l2":0},
    {"name":"O12-ClassWeight", "units":(64,32,16), "dropout":0.3, "lr":1e-3, "batch":32, "opt":"adam", "bn":False, "cw":True, "l2":0},
    {"name":"O13-L2", "units":(64,32,16), "dropout":0.3, "lr":1e-3, "batch":32, "opt":"adam", "bn":False, "cw":False, "l2":1e-4},
]
print("Number of experiments:", len(experiments))

## 4. Compute class weights from training data only

In [ ]:
n0 = np.sum(y_train_np == 0)
n1 = np.sum(y_train_np == 1)
total = n0 + n1
class_weights = {0: total/(2*n0), 1: total/(2*n1)}
print("Class counts:", {0:int(n0), 1:int(n1)})
print("Class weights:", class_weights)

## 5. Run experiments using validation metrics

In [ ]:
results = []
trained_models = {}

for cfg in experiments:
    print("Running:", cfg["name"])
    tf.keras.backend.clear_session()
    tf.random.set_seed(SEED)
    np.random.seed(SEED)

    model = build_dnn(
        X_train_np.shape[1], cfg["units"], cfg["dropout"],
        cfg["lr"], cfg["opt"], cfg["bn"], cfg["l2"]
    )

    cw = class_weights if cfg["cw"] else None
    model, history, elapsed = train_model(
        model, class_weight=cw, batch_size=cfg["batch"]
    )

    prob = probabilities(model, X_val_np)
    m = metrics(y_val_np, prob, 0.5)

    results.append({
        "experiment": cfg["name"],
        "architecture": "-".join(map(str, cfg["units"])),
        "dropout": cfg["dropout"],
        "learning_rate": cfg["lr"],
        "batch_size": cfg["batch"],
        "optimizer": cfg["opt"],
        "batch_norm": cfg["bn"],
        "class_weight": cfg["cw"],
        "l2": cfg["l2"],
        "parameters": model.count_params(),
        "epochs_trained": len(history.history["loss"]),
        "train_time_s": round(elapsed, 2),
        "val_accuracy": m["accuracy"],
        "val_precision": m["precision"],
        "val_recall": m["recall"],
        "val_f1": m["f1"],
        "val_roc_auc": m["roc_auc"]
    })
    trained_models[cfg["name"]] = model

results_df = pd.DataFrame(results).sort_values(
    ["val_f1", "val_roc_auc", "val_recall"],
    ascending=False
).reset_index(drop=True)

display(results_df)

## 6. Optimize the classification threshold on validation data

In [ ]:
# The threshold is selected using validation data only.
best_experiment = results_df.iloc[0]["experiment"]
best_model = trained_models[best_experiment]
val_prob = probabilities(best_model, X_val_np)

threshold_rows = []
for t in np.arange(0.20, 0.81, 0.01):
    m = metrics(y_val_np, val_prob, float(t))
    threshold_rows.append({"threshold":round(float(t),2), **m})

threshold_df = pd.DataFrame(threshold_rows)
best_threshold_row = threshold_df.sort_values(
    ["f1", "recall", "roc_auc"], ascending=False
).iloc[0]
best_threshold = float(best_threshold_row["threshold"])

print("Best experiment:", best_experiment)
print("Best validation threshold:", best_threshold)
display(threshold_df.sort_values("f1", ascending=False).head(10))

## 7. Final test evaluation — do this only after selection

In [ ]:
test_prob = probabilities(best_model, X_test_np)
test_metrics = metrics(y_test_np, test_prob, best_threshold)

print("FINAL OPTIMIZED DNN — TEST RESULTS")
print("----------------------------------")
for k in ["accuracy","precision","recall","f1","roc_auc"]:
    print(f"{k:10s}: {test_metrics[k]:.4f}")

## 8. Compare with the existing ML baselines

In [ ]:
baseline = pd.DataFrame([
    {"Model":"Logistic Regression","Accuracy":0.8055358410,"Precision":0.6572327044,"Recall":0.5588235294,"F1":0.6040462428,"ROC-AUC":0.8420083185},
    {"Model":"Decision Tree","Accuracy":0.7941802697,"Precision":0.6296296296,"Recall":0.5454545455,"F1":0.5845272206,"ROC-AUC":0.8283577463},
    {"Model":"Random Forest","Accuracy":0.7892122072,"Precision":0.6305084746,"Recall":0.4973262032,"F1":0.5560538117,"ROC-AUC":0.8225800202}
])

dnn_row = pd.DataFrame([{
    "Model":f"Optimized DNN ({best_experiment})",
    "Accuracy":test_metrics["accuracy"],
    "Precision":test_metrics["precision"],
    "Recall":test_metrics["recall"],
    "F1":test_metrics["f1"],
    "ROC-AUC":test_metrics["roc_auc"]
}])

comparison = pd.concat([baseline, dnn_row], ignore_index=True)
display(comparison)

lr = baseline.iloc[0]
dnn = dnn_row.iloc[0]

print("\nDNN minus Logistic Regression:")
for col in ["Accuracy","Precision","Recall","F1","ROC-AUC"]:
    print(f"{col:10s}: {dnn[col]-lr[col]:+.4f}")

## 9. Save final optimized artifacts

In [ ]:
models_dir = os.path.join("..", "models")
reports_dir = os.path.join("..", "reports")
os.makedirs(models_dir, exist_ok=True)
os.makedirs(reports_dir, exist_ok=True)

model_path = os.path.join(models_dir, "churn_dnn_optimized.keras")
preprocessor_path = os.path.join(models_dir, "preprocessor_optimized.joblib")

best_model.save(model_path)
joblib.dump(preprocessor, preprocessor_path)

results_df.to_csv(
    os.path.join(reports_dir, "dnn_optimization_experiments.csv"), index=False
)
comparison.to_csv(
    os.path.join(reports_dir, "dnn_optimized_comparison.csv"), index=False
)
threshold_df.to_csv(
    os.path.join(reports_dir, "dnn_threshold_results.csv"), index=False
)

metadata = {
    "selected_experiment": best_experiment,
    "validation_threshold": best_threshold,
    "test_metrics": {k: float(test_metrics[k]) for k in ["accuracy","precision","recall","f1","roc_auc"]},
    "target_mapping": {"No":0,"Yes":1},
    "model_path": model_path,
    "preprocessor_path": preprocessor_path
}

with open(os.path.join(reports_dir, "dnn_optimized_metadata.json"), "w") as f:
    json.dump(metadata, f, indent=2)

print("Saved optimized DNN:", model_path)
print("Saved preprocessor:", preprocessor_path)
print("Saved experiment/comparison/threshold reports.")

## 10. Handoff

If the optimized DNN is selected, give Mandara:

- `models/churn_dnn_optimized.keras`
- `models/preprocessor_optimized.joblib`
- `reports/dnn_optimization_experiments.csv`
- `reports/dnn_optimized_comparison.csv`
- `reports/dnn_threshold_results.csv`
- `reports/dnn_optimized_metadata.json`

If Logistic Regression still wins, keep these DNN results for the report and select the final model according to the agreed project metrics.

**Never tune the model again using the test results.**